In [1]:
# For the LLM, we recommend OpenAI with gpt-5.4-mini, but you can use any model and provider you like - 
# just adapt the client and the usage fields accordingly.

# Preparation
# First, we will pull the lesson pages straight from the course repository. We will use the commit 8c1834d to make sure 
# everyone works with the exact same data.

# We will use gitsource for that:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

# GithubRepositoryDataReader downloads the entire repository and goes over all the files in it. 
# Because we specify allowed_extensions={"md"}, it only checks the markdown files.


In [2]:
# We also pass a filename_filter so we don't grab every markdown file in the repo, like the top-level README. 
# The lesson pages all live under a module's lessons/ folder, so filtering on /lessons/ keeps just those.

# Each file has a parse() method that returns a dictionary with its filename and content:

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
# Q1. How many lesson pages
# How many lesson pages are in the dataset?

# 24
# 72 == 72
# 240
# 720

len(documents) # 72

72

In [4]:
documents[0]["filename"] # how to extract filename

'01-agentic-rag/lessons/01-intro.md'

In [5]:
# Q2. Indexing and searching
# Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

# << How does the agentic loop keep calling the model until it stops? >>

# What's the filename of the first result?

# 01-agentic-rag/lessons/03-rag.md
# 01-agentic-rag/lessons/14-agentic-loop.md == '01-agentic-rag/lessons/14-agentic-loop.md'
# 04-evaluation/lessons/13-llm-as-judge.md
# 06-best-practices/lessons/02-hybrid-search.md

from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
index

In [6]:
# Now search for our question from above:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    num_results=5
)

search_results[0]["filename"]
# '01-agentic-rag/lessons/14-agentic-loop.md'

'01-agentic-rag/lessons/14-agentic-loop.md'

In [25]:
# Q3. RAG
# Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

# wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
# RAGBase was written for the FAQ schema (section/question/answer), while our documents have filename and content.

# Two solutions are possible:

# Implement the RAG flow yourself
# Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
# Build a RAG over the index from Q2 and answer the query:

# How does the agentic loop keep calling the model until it stops?

# Use gpt-5.4-mini. How many input (prompt) tokens did we send to the model for this request?

# 700
# 7000 -- input_tokens=7193
# 70000
# 700000

# We count input tokens instead of price because the cost depends on the model and provider you use, 
# but the size of the prompt we send is the same for everyone.

# Most LLM APIs report token usage on the response object (e.g. response.usage.input_tokens / prompt_tokens). 
# We'll read the input tokens from there.

# You will need to modify the code for the rag helper to expose the usage.

# In the RAG Helper class, llm returns only the text. Modify it to return the whole response, 
# and change rag to return both the answer and usage (as a tuple or create a small dataclass for that).

# DEPLOYING MANUALLY

def search(question):
    boost_dict = {"question": 2.0, "section": 0.5}

    return index.search(
        question,
        boost_dict=boost_dict,
        num_results=5
    )

search_results = search(question)
# search_results  #[0]["filename"] # search works

In [23]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append("Q: " + question)
        lines.append("F: " + doc["filename"])
        lines.append("C: " + doc["content"])
        lines.append("")
        lines.append("+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++")
        lines.append("")

    return "\n".join(lines).strip()

# Now we combine the question with the context into the user prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()
    
# Let's try it:

prompt = build_prompt(question, search_results)

# print(prompt)
# We send it to the LLM:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=prompt
)

print(response.output_text)

The agentic loop keeps calling the model with a `while True` loop and stops only when the model returns **no function calls**.

### Core idea
1. Send the current `messages` to the model.
2. Inspect the response.
3. If the model requested a tool/function call:
   - run the tool,
   - append the tool result to `messages`,
   - loop again.
4. If there are **no** function calls in the response:
   - the model has finished,
   - `break` out of the loop.

### The stop condition
In the lesson code, that’s controlled by a flag like `has_function_calls`:

```python
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

if has_function_calls == False:
    break
```

So the loop ends when the model gives a final answer instead of asking for another tool call.

### In short
The model decides whether more searching is needed; your code just keeps repla

In [24]:
# The usage counts tell you how many tokens the request consumed:

response.usage # input_tokens=7193

ResponseUsage(input_tokens=7193, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=248, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7441)

In [12]:
# !!! Using RAGBase class from modified python file !!!

from openai import OpenAI
openai_client = OpenAI()

from rag_helper_costs import RAGBase
from ingest import load_faq_data, build_index

# documents = load_faq_data() # - we have done it all above - 72 docs we have in total
# index = build_index(documents)

# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    filename='01-agentic-rag/lessons/14-agentic-loop.md'
)

# Let's ask our question:

assistant.rag(question)

KeyError: 'section'